In [0]:
catalog = "venergy"
landing_schema = "landing"
bronze_schema = "bronze"
landing_volume = "landing_volume"
volume_path = f"/Volumes/{catalog}/{landing_schema}/{landing_volume}"

In [0]:
create_catalog = f"CREATE CATALOG IF NOT EXISTS {catalog}"
create_landing_schema = f"CREATE SCHEMA IF NOT EXISTS {catalog}.{landing_schema}"
create_bronze_schema = f"CREATE SCHEMA IF NOT EXISTS {catalog}.{bronze_schema}"
create_landing_volume = f"CREATE VOLUME IF NOT EXISTS {catalog}.{landing_schema}.{landing_volume}"

In [0]:
spark.sql(create_catalog)
spark.sql(create_landing_schema)
spark.sql(create_bronze_schema)
spark.sql(create_landing_volume)

In [0]:
sheets_df = spark.read.excel(
    path=f"{volume_path}/dados/inspecoes.xlsx",
    headerRows=1,
    listSheets=True,
)

sheet_names = [row.sheetName for row in sheets_df.collect()]

from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp, lit

dfs = []
for sheet in sheet_names:
    df_temp = (
        spark.read
        .option("headerRows", 1)
        .option("dataAddress", sheet)
        .excel(path=f"{volume_path}/dados/inspecoes.xlsx")
        .withColumn("sheet_name", lit(sheet))
    )
    dfs.append(df_temp)

df_inspecoes = dfs[0]
for df in dfs[1:]:
    df_inspecoes = df_inspecoes.unionByName(df)

df_inspecoes = df_inspecoes.withColumn("ingestion_datetime", current_timestamp())

df_inspecoes.write.format("delta").mode("append").saveAsTable(f"{catalog}.{bronze_schema}.inspecoes")